In [ ]:
# ==============================================================================
# 05_Semantic_Generalization_Defensive_Layer.ipynb
# Research Framework: Beyond Redaction
# Objective: Simulating Edge EdgeLLM Performance via Hosted google/gemma-4-31b-it
# ==============================================================================

# 1. Environment & Client Setup
print("[1/4] Configuring client for EdgeLLM simulation (google/gemma-4-31b-it)...")
import os
import json
from openai import OpenAI
from google.colab import userdata

try:
    NVIDIA_API_KEY = userdata.get('NVIDIA_API_KEY')
    client = OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=NVIDIA_API_KEY
    )
    print("--> Success: Proxy engine initialized.")
except Exception as e:
    print(f"[CRITICAL ERROR] Failed to load credentials: {e}")

# 2. Define the Edge Generalization Prompt Template
LOCAL_GENERALIZATION_PROMPT = """You are a localized on-device privacy proxy model running completely within an enterprise edge perimeter.
Your operational task is to transform incoming corporate text streams into a generalized, intent-preserving semantic structure.

STRICT PERFORMANCE CONSTRAINTS:
1. Strip all explicit corporate identifiers, internal system infrastructures, tool names, and specialized application labels.
2. Abstract specific ongoing internal projects or technical issues into generic, high-level functional descriptions (e.g., replace explicit portal or engine names with 'a core data processing module').
3. Convert concrete user roles, transactional paths, and networks into generalized operational layers.
4. Maintain 100% of the underlying business intent, technical logic, or functional request so downstream cloud services can execute correctly.

Return only the cleanly transformed text without any metadata tags, conversational filler, or structural labels."""

# 3. Access Verified Dataset and Prepare Corpus Snapshot for Edge Generalization
print("\n[2/4] Compiling target interaction history payload from verified corpus...")

import os
import tarfile
from google.colab import drive

# FORCED RESET: Wipe out any stale or empty variables lingering in the Colab runtime memory
sample_context = None

try:
    print("--> [INFO] Initializing clean storage check. Validating Google Drive paths...")

    # Ensure drive is explicitly mounted
    if not os.path.exists("/content/drive"):
        drive.mount('/content/drive')

    # Explicitly target both possible Colab Drive naming variations
    candidate_paths = [
        "/content/drive/MyDrive/beyond-redaction-data/enron_mail.tar.gz",
        "/content/drive/My Drive/beyond-redaction-data/enron_mail.tar.gz"
    ]

    TAR_FILE = None
    for path in candidate_paths:
        if os.path.exists(path):
            TAR_FILE = path
            print(f"--> Success! Archive firmly located at: {TAR_FILE}")
            break

    if not TAR_FILE:
        print("--> [DEBUG] Direct paths failed. Inspecting /content/drive contents:")
        if os.path.exists("/content/drive"):
            print(f"Available roots: {os.listdir('/content/drive')}")
        raise FileNotFoundError("Could not link to the archive using standard mount structures.")

    # Stream the 25 chronological records
    sample_records = []
    with tarfile.open(TAR_FILE, "r:gz") as tar:
        count = 0
        for member in tar:
            if member.isfile() and member.name.endswith('.'):
                f = tar.extractfile(member)
                if f is not None:
                    content = f.read().decode('utf-8', errors='ignore')
                    sample_records.append(content)
                    count += 1
            if count >= 25:
                break
    sample_context = "\n--- Next Communication Log ---\n".join(sample_records)
    print(f"--> Target Context Compiled from Storage: {len(sample_context.split()):,} words ready.")

except Exception as e:
    print(f"[CRITICAL ERROR] Failed to stage dataset text vectors: {e}")


# 4. Edge LLM Generalization via Gemma-4-31B-It (Reasoning Enabled)
print("\n[3/4] Running sanitized edge transformation via google/gemma-4-31b-it...")

import requests
import json
import re
import os
from google.colab import drive

# HARDENED PROMPT CONSTRAINTS (Self-Contained Definition)
HARDENED_EDGE_PROMPT = """You are a localized privacy proxy engine running on an enterprise edge device.
Your sole operational duty is to act as a text-to-text transformation filter.

CRITICAL CONSTRAINT: You must output ONLY the rewritten, generalized version of the user input text. Do not provide lists of recommendations, do not explain how to do it, do not write markdown checklists, and do not include conversational filler.

DIRECTIONS:
- Read the incoming corporate communications.
- Completely rewrite the narratives to strip explicit company names, application labels, and specific infrastructure titles.
- Transform explicit network topologies or employee role interactions into high-level abstract functional units.
- Keep the underlying operational logic, technical blockers, and business purpose completely intact so external cloud systems can successfully resolve the query.
"""

def clean_email_body(raw_text):
    """
    Strips raw Enron email headers and non-functional structural metadata
    to leave only the clean narrative text for SLM consumption.
    """
    lines = raw_text.split('\n')
    clean_lines = []
    in_headers = True

    for line in lines:
        if in_headers:
            if line.startswith('X-FileName:') or line.startswith('X-Folder:'):
                in_headers = False
            continue
        if re.match(r'^[=\-_*#\s\/\\]+$', line):
            continue
        clean_lines.append(line.strip())

    cleaned_body = " ".join([l for l in clean_lines if l]).strip()
    return " ".join(cleaned_body.split()[:200]) # Focused 200-word window per log

# Compile the target logs using the clean parser
sanitized_logs = []
for i, raw_email in enumerate(sample_records[:3], 1):
    cleaned = clean_email_body(raw_email)
    if cleaned:
        sanitized_logs.append(f"Document [{i}]:\n{cleaned}")

focused_sample_chunk = "\n\n".join(sanitized_logs)

# ------------------------------------------------------------------------------
# NVIDIA NIM API Request Execution via Requests
# ------------------------------------------------------------------------------
invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"

headers = {
    "Authorization": f"Bearer {NVIDIA_API_KEY}",
    "Content-Type": "application/json",
    "Accept": "application/json"
}

payload = {
    "model": "google/gemma-4-31b-it",
    "messages": [
        {
            "role": "user",
            "content": f"{HARDENED_EDGE_PROMPT}\n\nINPUT TEXT TO TRANSFORM:\n{focused_sample_chunk}"
        }
    ],
    "max_tokens": 2048,
    "temperature": 0.1,
    "top_p": 0.95,
    "stream": False,
    "chat_template_kwargs": {"enable_thinking": True},
}

# The execution block wrapped securely in a clean try-except boundary
if len(focused_sample_chunk.strip()) > 0:
    try:
        response = requests.post(invoke_url, headers=headers, json=payload)
        response_json = response.json()

        # Extract response text content from the response object
        generalized_context = response_json['choices'][0]['message']['content']

        print("\n==============================================================================")
        print("🛡️ EDGE PRIVACY SHIELD ACTIVE: GEMMA SEMANTIC GENERALIZATION COMPLETE 🛡️")
        print("==============================================================================")
        print(generalized_context[:1200] + "\n\n[... Clean Abstracted Profile Compiled ...]")

        # Ensure drive is explicitly attached to the current container session
        if not os.path.exists("/content/drive"):
            drive.mount('/content/drive')

        # Target the persistent cache folder directly
        PERSISTENT_CACHE_DIR = "/content/drive/MyDrive/beyond-redaction-data"
        os.makedirs(PERSISTENT_CACHE_DIR, exist_ok=True)
        OUTPUT_PATH = os.path.join(PERSISTENT_CACHE_DIR, "protected_generalized_context.txt")

        # Cache the clean artifact to persistent cloud storage
        with open(OUTPUT_PATH, 'w') as f:
            f.write(generalized_context)
            print(f"\n--> Output permanently cached to Google Drive: {OUTPUT_PATH}")

    except Exception as e:
        print(f"--> [ERROR] Gemma edge simulation pipeline failed: {e}")
        if 'response_json' in locals():
            print(f"--> API Error Details: {json.dumps(response_json, indent=2)}")
else:
    print("[ABORTED] Inference skipped because sample chunk is empty.")